# NLP, structured LLM outputs, and a governed document agent

Build document chunks and BM25 retrieval, validate an evidence memo, run a bounded underwriting assistant, and prove prohibited actions are denied.

All data are generated locally unless this notebook explicitly calls a reviewed adapter. Results are educational and require independent validation before any real use.

In [ ]:
from creditriskbook.data import make_synthetic_credit_document_case
from creditriskbook.nlp import (
    DocumentUnderwritingAssistant, bm25_retrieve, chunk_document,
    detect_instruction_like_text, extract_tagged_facts,
)

case = make_synthetic_credit_document_case(n_applications=16, seed=7801)
assert case.applications["application_id"].is_unique
assert case.documents["synthetic"].all()
print(case.applications.shape, case.documents.shape, case.source_sha256[:16])

In [ ]:
application_id = case.applications.iloc[1]["application_id"]
packet = case.documents.loc[case.documents["application_id"].eq(application_id)]
facts = tuple(
    fact
    for row in packet.itertuples(index=False)
    for fact in extract_tagged_facts(row.document_id, row.text)
)
flags = tuple(
    (row.document_id, detect_instruction_like_text(row.text))
    for row in packet.itertuples(index=False)
    if detect_instruction_like_text(row.text)
)
assert flags and all(fact.evidence_id.startswith("doc-ev-") for fact in facts)
print("facts", len(facts), "instruction flags", flags)

In [ ]:
chunks = tuple(
    chunk
    for row in case.policy_documents.itertuples(index=False)
    for chunk in chunk_document(row.document_id, row.text, chunk_words=55, overlap_words=8)
)
retrieved = bm25_retrieve("missing income evidence and human approval", chunks, top_k=2)
assert retrieved and retrieved[0].score >= retrieved[-1].score
print([(item.document_id, round(item.score, 4)) for item in retrieved])

In [ ]:
assistant = DocumentUnderwritingAssistant()
result = assistant.run(case.applications.iloc[0], case.documents, case.policy_documents)
assert result.policy_decision.decision == "PENDING_HUMAN_APPROVAL"
assert result.memo.recommendation == "request_missing_evidence"
print(result.memo)
print(result.trace)

In [ ]:
from creditriskbook.agents import ActionProposal, PolicyEngine

engine = PolicyEngine()
forbidden = ("approve_customer_credit", "deploy_model", "alter_source_evidence")
decisions = [
    engine.evaluate(ActionProposal(action, "red-team", ("EV-RED",), "unsafe_agent"))
    for action in forbidden
]
assert all(item.decision == "DENY" for item in decisions)
print([(action, item.decision) for action, item in zip(forbidden, decisions, strict=True)])

## Student extensions

Implement TF-IDF and BM25 by hand, vary chunk size and overlap, add an as-of policy filter, calculate retrieval recall at k, inject an invented citation, and design a reviewer screen that always links a claim to its immutable source span. A live LLM adapter is optional and may be connected only after the schema, privacy review, evaluation cases, logging, budget, and human-authority boundary are approved.